In [25]:
import pandas as pd # manipulação de tabelas
import openpyxl # leitura e escrita de arquivos Excel
import requests # chamadas de APIs
import json # para tratar a resposta das APIs
import concurrent.futures # Para paralelizar as requisições
from time import sleep # para dar um tempo de segurança entre requisições
from chembl_webresource_client.new_client import new_client # chamar API do CheBl
import re # regex para identificação rápida de certas moléculas
import urllib.parse # para codificar nomes de moléculas em URLs

# 1. Mantém conexões TCP/SSL ativas (economiza ~0.3s por chamada)
session = requests.Session()

# 2. Instanciação Global do ChEMBL: Evita reconfigurar o cliente a cada molécula
chembl_molecule = new_client.molecule

CRIAÇÃO E MANIPULAÇÃO DOS ARQUIVOS PRINCIPAIS

In [ ]:
# leitura das planilhas
df_ident = pd.read_excel('tabelas_principais/IDENTIFICACAO.xlsx', usecols=['Compound', 'Compound ID', 'Formula', 'Fragmentation Score', 'Score', 'Isotope Similarity', 'Description', 'm/z'])
df_abund = pd.read_excel('tabelas_principais/ABUND.xlsx', usecols=['Compound', 'Identifications'])

df = pd.merge(df_ident, df_abund, on='Compound') # merge das planilhas
df = df.dropna(subset=['Description']) # retira linhas a qual não tenha nome (impossibilita pesquisa)

PESQUISA NAS APIS

In [27]:
def Pubchem_search(name: str):
    result = {'Information': 'NaN', 'Molecular_Uses': 'NaN', 'Info_Source': 'NaN', 'IUPAC Name': 'NaN', 'SMILES': 'NaN', 'InChIKey': 'NaN'}
    url_props = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/IUPACName,SMILES,InChIKey/JSON'
    try:
        res_props = session.get(url_props, timeout=4)
        if res_props.status_code == 200:
            props = res_props.json()['PropertyTable']['Properties'][0]
            result['IUPAC Name'] = props.get('IUPACName', 'NaN')
            result['SMILES'] = props.get('SMILES', 'NaN')
            result['InChIKey'] = props.get('InChIKey', 'NaN')
            
            url_desc = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/description/JSON'
            res_desc = session.get(url_desc, timeout=4)
            
            if res_desc.status_code == 200:
                data_list = res_desc.json()['InformationList']['Information']
                descricoes = []
                usos = []
                for item in data_list:
                    if 'Description' in item:
                        text = item.get('Description')
                        source = item.get('DescriptionSourceName')
                        keywords_uso = ['used for', 'use in', 'application', 'treatment', 'therapeutic', 'industry']
                        if any(key in text.lower() for key in keywords_uso):
                            usos.append(text)
                        else:
                            descricoes.append(text)
                            result['Info_Source'] = source

                if descricoes: result['Information'] = descricoes[0]
                if usos: result['Molecular_Uses'] = " | ".join(usos[:2])
    except Exception:
        pass
    return result

In [28]:
def Chembl_search(inchikey: str = None, name: str = None):
    try:
        # Só gasta tempo no ChEMBL se tivermos o InChIKey exato.
        # Buscar por nomes complexos no ChEMBL gera travamentos severos na API deles.
        if inchikey and inchikey != 'NaN' and not pd.isna(inchikey):
            res = chembl_molecule.filter(molecule_structures__standard_inchi_key=inchikey).only([
                'molecule_type', 'natural_product', 'max_phase', 'structure_type', 'molecule_hierarchy'
            ])
            if res:
                mol = res[0]
                return {
                    'ChEMBL_Type': mol.get('molecule_type'),
                    'Natural_Product': 'Yes' if mol.get('natural_product') == 1 else 'No',
                    'Max_Clinical_Phase': mol.get('max_phase'),
                    'Is_Parent': 'Yes' if mol.get('molecule_hierarchy', {}).get('parent_chembl_id') == mol.get('molecule_chembl_id') else 'No'
                }
    except Exception:
        pass
    return {}

In [29]:
def HMDB_search(inchikey: str = None, name: str = None):
    base_url = "https://mychem.info/v1/query"
    
    # Passamos os argumentos via dicionário 'params' para o requests automatizar o URL Encode
    if inchikey and inchikey != 'NaN' and not pd.isna(inchikey):
        params = {'q': f'inchikey:{inchikey}', 'fields': 'hmdb'}
    elif name:
        params = {'q': f'name:"{name}"', 'fields': 'hmdb'}
    else:
        return {'Is_Human_Metabolite': 'No', 'HMDB_ID': 'NaN'}
    
    try:
        response = session.get(base_url, params=params, timeout=2)
        if response.status_code == 200:
            data = response.json()
            if data.get('hits') and len(data['hits']) > 0 and 'hmdb' in data['hits'][0]:
                hmdb_data = data['hits'][0]['hmdb']
                if isinstance(hmdb_data, list):
                    hmdb_data = hmdb_data[0]
                    
                return {
                    'Is_Human_Metabolite': 'Yes',
                    'HMDB_ID': hmdb_data.get('accession', 'Found')
                }
    except Exception:
        pass
    return {'Is_Human_Metabolite': 'No', 'HMDB_ID': 'NaN'}

In [30]:
def FooDB_search(inchikey: str = None, name: str = None):
    base_url = "https://biothings.ncats.io/foodb/query"
    
    if inchikey and inchikey != 'NaN' and not pd.isna(inchikey):
        params = {'q': f'inchikey:{inchikey}'}
    elif name:
        params = {'q': f'name:"{name}"'}
    else:
        return {'In_FooDB': 'No', 'FooDB_ID': 'NaN'}
        
    try:
        response = session.get(base_url, params=params, timeout=2)
        if response.status_code == 200:
            data = response.json()
            if data.get('hits') and len(data['hits']) > 0:
                foodb_id = data['hits'][0].get('_id', 'Found')
                return {
                    'In_FooDB': 'Yes',
                    'FooDB_ID': foodb_id
                }
    except Exception:
        pass
    return {'In_FooDB': 'No', 'FooDB_ID': 'NaN'}

AÇÕES PRINCIPAIS

In [31]:
# explicação da função:
"""
Verifica se a molécula cai em alguma regra de nomenclatura local 
antes de gastar tempo buscando na internet.
Na metabolômica, quando esbarramos nesses peptídeos quebrados, geralmente não precisamos da estrutura química exata
(SMILES/InChIKey) deles. O que a gente precisa é que a tabela final não ache que isso é um "Sintético não encontrado",
mas sim que classifique como um Peptídeo Natural.
"""

def Regras_Locais_search(name: str):
    name = str(name).strip()
    
    # Regra 1: Peptídeos (ex: ala-arg-ile-pro)
    if re.match(r'^([a-zA-Z]{3,4}-)+[a-zA-Z]{3,4}$', name):
        return {
            'InChIKey': 'NaN', 'IUPAC Name': 'NaN', 'SMILES': 'NaN', 
            'Information': 'Fragmento de peptídeo identificado por sequenciamento.', 
            'Molecular_Uses': 'NaN', 'Info_Source': 'Regras Locais (Peptídeo)',
            'ChEMBL_Type': 'Oligopeptide', 'Natural_Product': 'Yes',
            'Max_Clinical_Phase': 'NaN', 'Is_Human_Metabolite': 'Yes', 
            'HMDB_ID': 'NaN', 'In_FooDB': 'Yes', 'FooDB_ID': 'NaN'
        }
        
    # Se não cair em nenhuma regra, retorna None para avisar o loop que tem que usar as APIs
    return None

In [32]:
def classificar_natureza(row):
    is_metabolite = row.get('Is_Human_Metabolite')
    natural_chembl = row.get('Natural_Product')
    fase_clinica = row.get('Max_Clinical_Phase')
    info_pubchem = str(row.get('Information', '')).lower()
    
    if is_metabolite == 'Yes': 
        return 'Metabólito Endógeno (HMDB)'
    if natural_chembl == 'Yes': 
        return 'Produto Natural'
    if 'metabolite' in info_pubchem and is_metabolite != 'Yes': 
        return 'Metabólito (Não-humano / Geral)'
    if pd.notna(fase_clinica) and fase_clinica == 4: 
        return 'Fármaco Sintético Aprovado'
    if pd.notna(fase_clinica) and isinstance(fase_clinica, (int, float)) and fase_clinica > 0:
        return f'Sintético em Investigação (Fase {int(fase_clinica)})'
    return 'Sintético / Indefinido'

In [33]:
def processar_uma_molecula(item):
    index, linha_original = item
    compound_name = linha_original['Description']
    
    try:
        # 1. Regra Local
        busca_local = Regras_Locais_search(compound_name)
        if busca_local is not None:
            return {**linha_original, **busca_local}

        # 2. Busca PubChem
        search_pubchem = Pubchem_search(compound_name)
        inchikey = search_pubchem.get('InChIKey')

        # 3. Demais APIs sequenciais
        search_chembl = Chembl_search(inchikey, compound_name)
        search_hmdb = HMDB_search(inchikey, compound_name)
        search_foodb = FooDB_search(inchikey, compound_name)

        linha_combinada = {**linha_original, **search_pubchem, **search_chembl, **search_hmdb, **search_foodb}
        print(f"[{index}] {compound_name[:25]}... ➔ Enriquecido!")
        
        # Aumentado levemente para 0.15s para dar fôlego ao Rate Limit do PubChem em execuções longas
        sleep(0.15)
        return linha_combinada

    except Exception as e:
        print(f"[{index}] ERRO CRÍTICO em {compound_name[:20]}: {e}")
        return linha_original

In [ ]:
# ==============================================================================
# EXECUTOR PRINCIPAL OTIMIZADO (Trata duplicatas e corrige perda de linhas)
# ==============================================================================
import time
import os
import json
import csv
import concurrent.futures
import pandas as pd

# Lista final de colunas exigida pelo IST Blumenau
colunas_desejadas = ['Compound ID', 'Description', 'Natureza_Final', 'Natural_Product', 'Formula', 'SMILES', 'IUPAC Name', 'Score', 'Fragmentation Score', 'Isotope Similarity', 'Identifications', 'Information', 'Molecular_Uses', 'ChEMBL_Type', 'Max_Clinical_Phase', 'Is_Human_Metabolite', 'In_FooDB', 'Info_Source', 'm/z', 'InChIKey', 'HMDB_ID', 'FooDB_ID']

df_teste = df

# 1. Pegar apenas as descrições únicas para evitar milhares de buscas repetidas
descricoes_unicas = df_teste['Description'].dropna().unique()
print(f"Linhas totais no dataframe: {len(df_teste)}")
print(f"Descrições únicas a serem buscadas nas APIs: {len(descricoes_unicas)}\n")

checkpoint_file = 'dicionario_api_checkpoint.csv'
time_log_file = 'execution_time_log.json'

total_previous_time = 0.0
if os.path.exists(time_log_file):
    try:
        with open(time_log_file, 'r') as f:
            total_previous_time = json.load(f).get('total_time_seconds', 0.0)
    except:
        pass

# Carregar checkpoint como um dicionário
processados_dict = {}
if os.path.exists(checkpoint_file):
    try:
        df_check = pd.read_csv(checkpoint_file)
        if 'Description' in df_check.columns:
            # Substituir NaN string literals por string vazia ou np.nan, mas manter tudo normal
            processados_dict = df_check.set_index('Description').to_dict('index')
            print(f"✅ Checkpoint encontrado! {len(processados_dict)} moléculas únicas já processadas.")
    except Exception as e:
        print(f"⚠️ Erro ao ler checkpoint: {e}")

# Preparar tarefas pendentes
descricoes_pendentes = [desc for desc in descricoes_unicas if desc not in processados_dict]
# Criar uma lista de tuplas (idx, dict) para reusar a função processar_uma_molecula
tarefas_pendentes = [(i, {'Description': desc}) for i, desc in enumerate(descricoes_pendentes)]

n_de_pedidos = 12 # Reduzido um pouco para não levar ban da API por muitas requisições simultâneas

print(f"\n🚀 Iniciando processamento paralelo de {len(tarefas_pendentes)} moléculas únicas faltantes...\n")

start_time = time.time()

try:
    if tarefas_pendentes:
        with concurrent.futures.ThreadPoolExecutor(max_workers=n_de_pedidos) as executor:
            future_to_tarefa = {executor.submit(processar_uma_molecula, tarefa): tarefa for tarefa in tarefas_pendentes}
            
            for future in concurrent.futures.as_completed(future_to_tarefa):
                resultado = future.result()
                desc = resultado['Description']
                
                # O resultado traz 'Description' e as chaves das APIs, mas falta colunas para salvar
                # Salvamos um dict simples apenas com o que achou nas apis
                res_df = pd.DataFrame([resultado])
                
                header_flag = not os.path.exists(checkpoint_file)
                res_df.to_csv(checkpoint_file, mode='a', index=False, header=header_flag, lineterminator='\n')
                
                # Salva na memória também
                processados_dict[desc] = resultado
                
except KeyboardInterrupt:
    print("\n🛑 PROCESSAMENTO INTERROMPIDO PELO USUÁRIO.")
except Exception as e:
    print(f"\n❌ ERRO DURANTE O PROCESSAMENTO: {e}")
finally:
    end_time = time.time()
    session_time = end_time - start_time
    total_time = total_previous_time + session_time
    
    with open(time_log_file, 'w') as f:
        json.dump({'total_time_seconds': total_time}, f)
        
    def format_time(seconds):
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = int(seconds % 60)
        return f"{h}h {m}m {s}s"
        
    print(f"\n⏱️ Tempo desta sessão: {format_time(session_time)}")
    print(f"🕒 Tempo total acumulado: {format_time(total_time)}")

# Se terminamos todas as descrições únicas, podemos reconstruir o dataframe completo!
if len(processados_dict) >= len(descricoes_unicas):
    print("\n⏳ Todas as APIs concluídas! Mesclando resultados com as 37 mil linhas...")
    
    # 1. Carregar o checkpoint limpo
    df_apis = pd.read_csv(checkpoint_file)
    # remover duplicatas do checkpoint por garantia
    df_apis = df_apis.drop_duplicates(subset=['Description'], keep='last')
    
    # 2. Fazer o merge do df_teste (original) com os dados enriquecidos
    # Como df_teste e df_apis podem ter colunas em comum (Description), apagamos do df_teste as que as apis sobrescrevem se houver
    colunas_apis_exceto_desc = [c for c in df_apis.columns if c != 'Description']
    df_teste_limpo = df_teste.drop(columns=[c for c in colunas_apis_exceto_desc if c in df_teste.columns], errors='ignore')
    
    df_final = pd.merge(df_teste_limpo, df_apis, on='Description', how='left')
    
    # 3. Aplicar classificar_natureza
    df_final['Natureza_Final'] = df_final.apply(classificar_natureza, axis=1)
    
    # 4. Ajustar colunas
    colunas_presentes = [col for col in colunas_desejadas if col in df_final.columns]
    df_final = df_final.reindex(columns=colunas_presentes).fillna('NaN')
    
    # 5. Salvar o arquivo final com TODAS as linhas
    df_final.to_csv('final_integrado_com_hmdb.csv', index=False, lineterminator='\n')
    print(f"✨ Concluído! Arquivo 'final_integrado_com_hmdb.csv' gerado com sucesso com {len(df_final)} linhas!")



🚀 Iniciando processamento paralelo de 1500 moléculas...

[53] 8-Chloro-5-(β-L-glucopyra... ➔ Enriquecido!
[70] AL 8810 ethyl amide... ➔ Enriquecido!
[65] 7-(2-fluoro-4-methoxyphen... ➔ Enriquecido!
[55] Adozelesin... ➔ Enriquecido!
[222] Tofacitinib citrate... ➔ Enriquecido!
[52] (1R,2S,3R)-2-(2,4-Dihydro... ➔ Enriquecido!
[72] Mifepristone... ➔ Enriquecido!
[224] 2-[(E)-2-(2-Naphthyl)viny... ➔ Enriquecido!
[56] 5-[8-Fluoro-2-({[2-(1H-1,... ➔ Enriquecido!
[58] (2E)-4-{3-[(2E)-3-(2,4-Di... ➔ Enriquecido!
[64] N-(2,4-difluorophenyl)-3-... ➔ Enriquecido!
[63] 7-(4-fluoro-2-methoxyphen... ➔ Enriquecido!
[223] 2-(2,4-Difluorophenoxy)-1... ➔ Enriquecido!
[235] 2,6-Pyridinedicarboxylic ... ➔ Enriquecido!
[232] 2-(1,5-dimethyl-4-piperid... ➔ Enriquecido!
[74] 7,7,9,9-Tetramethyl-3-[3-... ➔ Enriquecido!
[231] N-[2-(dimethylamino)ethyl... ➔ Enriquecido!
[221] Cryptospirolepine... ➔ Enriquecido![54] Adozelesin... ➔ Enriquecido!

[236] N-{3-[(4,6-Diamino-1,3,5-... ➔ Enriquecido!
[71] Miproxifene.